In [7]:
#import packages
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from pathlib import Path

In [8]:
# Build the path to the CSV file
notebook_dir = Path.cwd()
csv_file = notebook_dir / 'refinitiv_data' / 'nvda_options_20260204_210456.csv'
df=pd.read_csv(csv_file)

In [9]:
# Rename columns
df.columns = ['instrument', 'iv', 'strike', 'expiry', 'bid', 'ask', 'last', 'type_option']

# Clean and prepare data
df['expiry'] = pd.to_datetime(df['expiry'])
df['type_option'] = df['type_option'].str.strip()
df['bid_ask_spread'] = (df['ask'] - df['bid']) / (0.5 * (df['bid'] + df['ask']))
df = df.dropna()

# Apply liquidity filter
liq_threshold = 2  # High threshold to include OTM puts
df_liq = df[df['bid_ask_spread'] <= liq_threshold].copy()

# Constants
S = 174.19  # NVDA spot price (Feb 4, 2026)
capture_date = pd.to_datetime('2026-02-04')

# Calculate time to expiry (in years)
df_liq['T'] = (df_liq['expiry'] - capture_date).dt.days / 365.25

# Risk-free rate function
def get_risk_free_rate(T):
    """US Treasury rates as of Feb 6, 2026"""
    if T < 0.25:
        return 0.03598   # 3.598% (US3MT)
    elif T < 0.5:
        return 0.03513   # 3.513% (US6MT)
    elif T < 1.0:
        return 0.03328   # 3.328% (US1YT)
    else:
        return 0.0325    # 3.25% (US2YT estimated)

# Apply risk-free rate (use T, not expiry!)
df_liq['r'] = df_liq['T'].apply(get_risk_free_rate)

# Calculate forward prices
df_liq['forward'] = S * np.exp(df_liq['r'] * df_liq['T'])

# Calculate moneyness measures
df_liq['moneyness'] = df_liq['strike'] / df_liq['forward']
df_liq['log_moneyness'] = np.log(df_liq['strike'] / df_liq['forward'])

# Calculate total variance (for SVI)
df_liq['total_variance'] = (df_liq['iv'] / 100) ** 2 * df_liq['T']

# Separate calls and puts
df_liq_calls = df_liq[df_liq['type_option'] == 'CALL'].copy()
df_liq_puts = df_liq[df_liq['type_option'] == 'PUT'].copy()

# Summary
print("Data Summary:")
print(f"Total options: {len(df_liq)}")
print(f"  Calls: {len(df_liq_calls)}")
print(f"  Puts: {len(df_liq_puts)}")
print(f"\nExpiry range: {df_liq['expiry'].min().date()} to {df_liq['expiry'].max().date()}")
print(f"Time to expiry range: {df_liq['T'].min():.3f} to {df_liq['T'].max():.3f} years")
print(f"Forward price range: ${df_liq['forward'].min():.2f} to ${df_liq['forward'].max():.2f}")

df_liq.describe()

Data Summary:
Total options: 991
  Calls: 503
  Puts: 488

Expiry range: 2026-02-06 to 2028-12-15
Time to expiry range: 0.005 to 2.861 years
Forward price range: $174.22 to $191.16


,iv,strike,expiry,bid,ask,last,bid_ask_spread,T,r,forward,moneyness,log_moneyness,total_variance
count,991.000000,991.000000,991,991.000000,991.000000,991.000000,991.000000,991.000000,991.000000,991.000000,991.000000,991.000000,991.000000
mean,75.231479,147.752775,2026-10-10 03:55:23.915237,40.140373,40.754753,40.655237,0.201506,0.679435,0.034372,178.215118,0.829648,-0.553698,0.434697
min,38.617900,0.500000,2026-02-06 00:00:00,0.000000,0.010000,0.010000,0.001142,0.005476,0.032500,174.224322,0.002789,-5.882178,0.001158
25%,46.170050,95.000000,2026-05-15 00:00:00,5.550000,5.625000,5.490000,0.006715,0.273785,0.033280,175.873454,0.543023,-0.610604,0.074540
50%,48.917900,165.000000,2026-06-18 00:00:00,24.550000,24.650000,24.710000,0.010949,0.366872,0.035130,176.449527,0.918564,-0.084944,0.177307
75%,76.889400,195.000000,2027-01-15 00:00:00,51.425000,51.975000,51.025000,0.025316,0.944559,0.035130,179.752619,1.087863,0.084215,0.442105
max,610.421700,380.000000,2028-12-15 00:00:00,203.550000,205.350000,199.650000,2.000000,2.861054,0.035980,191.163850,2.178090,0.778448,13.670188
std,57.829365,80.659193,NaN,47.389229,48.455843,48.629794,0.545530,0.643661,0.001307,3.760714,0.455728,1.174048,0.933782
